# 01 — Data Exploration

**Purpose:** Understand what's in the data before building anything.  
**Questions we answer here:**
- How big is the dataset? What date range does it cover?
- What are the top categories and destinations by spend?
- What does the customer demographic profile look like?
- Are there data quality issues (nulls, outliers, duplicates)?

**Tables used:** `staging.stg_transactions`, `staging.stg_customers`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px

PROJECT = 'fmn-sandbox'
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Dataset overview
How many transactions, customers, and what date range?

In [ ]:
overview = q(f"""
    SELECT
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT UNIQUE_ID) AS unique_customers,
        MIN(EFF_DATE) AS earliest_date,
        MAX(EFF_DATE) AS latest_date,
        DATE_DIFF(MAX(EFF_DATE), MIN(EFF_DATE), MONTH) AS months_of_data,
        ROUND(SUM(trns_amt), 0) AS total_spend,
        ROUND(AVG(trns_amt), 2) AS avg_transaction,
        COUNT(DISTINCT CATEGORY_TWO) AS distinct_categories,
        COUNT(DISTINCT DESTINATION) AS distinct_destinations,
        COUNT(DISTINCT PROVINCE) AS distinct_provinces
    FROM `{PROJECT}.staging.stg_transactions`
""")
overview.T

## 2. Top 20 categories by spend
Which CATEGORY_TWO groupings carry the most value?

In [ ]:
top_cats = q(f"""
    SELECT CATEGORY_TWO,
           COUNT(*) AS transactions,
           COUNT(DISTINCT UNIQUE_ID) AS customers,
           ROUND(SUM(trns_amt), 0) AS total_spend,
           ROUND(AVG(trns_amt), 2) AS avg_txn
    FROM `{PROJECT}.staging.stg_transactions`
    WHERE CATEGORY_TWO IS NOT NULL
    GROUP BY CATEGORY_TWO
    ORDER BY total_spend DESC
    LIMIT 20
""")

fig = px.bar(top_cats, x='total_spend', y='CATEGORY_TWO', orientation='h',
             color='customers', color_continuous_scale='Blues',
             title='Top 20 categories by total spend')
fig.update_layout(yaxis=dict(autorange='reversed'), height=600)
fig.show()

## 3. Top 20 destinations by spend
Which specific merchants generate the most revenue?

In [ ]:
top_dest = q(f"""
    SELECT DESTINATION, CATEGORY_TWO,
           COUNT(DISTINCT UNIQUE_ID) AS customers,
           ROUND(SUM(trns_amt), 0) AS total_spend,
           ROUND(AVG(trns_amt), 2) AS avg_txn
    FROM `{PROJECT}.staging.stg_transactions`
    WHERE DESTINATION IS NOT NULL
    GROUP BY DESTINATION, CATEGORY_TWO
    ORDER BY total_spend DESC
    LIMIT 20
""")

fig = px.bar(top_dest, x='total_spend', y='DESTINATION', orientation='h',
             color='CATEGORY_TWO', title='Top 20 destinations by spend')
fig.update_layout(yaxis=dict(autorange='reversed'), height=600)
fig.show()

## 4. Customer demographics
What does the customer base look like?

In [ ]:
demo = q(f"""
    SELECT
        COUNT(*) AS total_customers,
        ROUND(AVG(age), 1) AS avg_age,
        ROUND(AVG(estimated_income), 0) AS avg_income,
        COUNTIF(gender = 0) AS males,
        COUNTIF(gender = 1) AS females,
        COUNTIF(main_banked = 1) AS main_banked,
        COUNTIF(age IS NULL) AS null_age,
        COUNTIF(estimated_income IS NULL) AS null_income
    FROM `{PROJECT}.staging.stg_customers`
""")
demo.T

In [ ]:
# Age distribution
age_dist = q(f"""
    SELECT age_group, COUNT(*) AS customers
    FROM `{PROJECT}.staging.stg_customers`
    WHERE age_group != 'Unknown'
    GROUP BY age_group ORDER BY age_group
""")

fig = px.bar(age_dist, x='age_group', y='customers', title='Customer age distribution')
fig.show()

In [ ]:
# Income distribution
income_dist = q(f"""
    SELECT income_group, COUNT(*) AS customers
    FROM `{PROJECT}.staging.stg_customers`
    WHERE income_group != 'Unknown'
    GROUP BY income_group ORDER BY income_group
""")

fig = px.bar(income_dist, x='income_group', y='customers', title='Customer income distribution')
fig.show()

## 5. Data quality checks
Null rates across key fields, potential duplicates.

In [ ]:
nulls = q(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNTIF(UNIQUE_ID IS NULL) AS null_unique_id,
        COUNTIF(EFF_DATE IS NULL) AS null_eff_date,
        COUNTIF(trns_amt IS NULL) AS null_trns_amt,
        COUNTIF(CATEGORY_TWO IS NULL) AS null_category,
        COUNTIF(DESTINATION IS NULL) AS null_destination,
        COUNTIF(PROVINCE IS NULL) AS null_province,
        COUNTIF(MUNICIPALITY IS NULL) AS null_municipality
    FROM `{PROJECT}.staging.stg_transactions`
""")

total = nulls['total_rows'].iloc[0]
null_pcts = nulls.drop(columns='total_rows').T.reset_index()
null_pcts.columns = ['field', 'null_count']
null_pcts['null_pct'] = round(null_pcts['null_count'] / total * 100, 2)
null_pcts

## 6. Monthly transaction volume
Is the data evenly distributed or are there gaps?

In [ ]:
monthly = q(f"""
    SELECT DATE_TRUNC(EFF_DATE, MONTH) AS month,
           COUNT(*) AS transactions,
           COUNT(DISTINCT UNIQUE_ID) AS customers,
           ROUND(SUM(trns_amt), 0) AS spend
    FROM `{PROJECT}.staging.stg_transactions`
    GROUP BY month ORDER BY month
""")

fig = px.line(monthly, x='month', y='transactions', title='Monthly transaction volume')
fig.show()